In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

np.random.seed(42)

# File paths

SPLIT_DIR = Path("../backend/data/splits_70_15_15")
OUT_DIR = Path("../backend/data/EKF_Outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = SPLIT_DIR / "train.parquet"
VAL_PATH = SPLIT_DIR / "val.parquet"
TEST_PATH = SPLIT_DIR / "test.parquet"

MODEL_TAG = "EKF"

# Configuration

OBS_COLS = ["temperature", "humidity", "audio_density"]
D = len(OBS_COLS)

S_JITTER = 1e-6
MIN_ROWS_VAL = 200
MIN_ROWS_TEST = 200

Q_SCALES = [0.01, 0.03, 0.1, 0.3, 1.0, 3.0]

# EKF nonlinearity strength
ALPHA = 0.02

# State prior covariance
P0 = np.eye(D) * 10.0

# Time-gap-aware scaling for process noise Q (to align with the KF fairness approach)
BASE_DT_MIN = 15.0
USE_DT_AWARE_Q = True
DT_SCALE_MODE = "since_observed"  
DT_CLIP_MIN = 1.0
DT_CLIP_MAX = 16.0

# Load and clean data

def load_and_clean(path: Path) -> pd.DataFrame:
    """
    Load a parquet file and standardize types.
    Ensures:
      - published_at is UTC datetime
      - tag_number is integer-like
      - observation columns are numeric
    """
    df = pd.read_parquet(path).copy()
    df["published_at"] = pd.to_datetime(df["published_at"], utc=True, errors="coerce")
    df["tag_number"] = pd.to_numeric(df["tag_number"], errors="coerce").astype("Int64")
    for c in OBS_COLS:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=["published_at", "tag_number"])
    df = df.sort_values(["tag_number", "published_at"]).reset_index(drop=True)
    return df


# Utility functions

def persistence_missing_safe(z: np.ndarray) -> np.ndarray:
    """
    Baseline forecast: persistence using the last observed value per dimension.
    Produces NaN until the first observation arrives for that dimension.
    """
    T, d = z.shape
    out = np.full((T, d), np.nan, float)
    last = np.full((d,), np.nan, float)
    for t in range(T):
        out[t] = last
        m = np.isfinite(z[t])
        last[m] = z[t, m]
    return out

def rmse_per_dim(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    """Compute RMSE per dimension, ignoring NaNs."""
    out = np.full((y_true.shape[1],), np.nan, float)
    for i in range(y_true.shape[1]):
        m = np.isfinite(y_true[:, i]) & np.isfinite(y_pred[:, i])
        if m.sum() > 0:
            out[i] = float(np.sqrt(np.mean((y_true[m, i] - y_pred[m, i]) ** 2)))
    return out

def rows_with_any_obs(df_h: pd.DataFrame) -> int:
    """Count rows with at least one finite observation across OBS_COLS."""
    z = df_h[OBS_COLS].to_numpy(float)
    return int(np.isfinite(z).any(axis=1).sum())

def init_x0_from_first_valid(z: np.ndarray) -> np.ndarray:
    """Initialize the state with the first available observation per dimension."""
    x0 = np.zeros((D,), float)
    for t in range(len(z)):
        m = np.isfinite(z[t])
        if m.any():
            x0[m] = z[t, m]
            break
    return x0

def percentile_threshold(arr, p=95.0, min_n=200):
    """Return the p-th percentile threshold if there are at least min_n finite samples."""
    a = np.asarray(arr, float)
    a = a[np.isfinite(a)]
    if a.size < min_n:
        return np.nan
    return float(np.percentile(a, p))

def sym(A: np.ndarray) -> np.ndarray:
    """Symmetrize a square matrix to reduce numerical asymmetry."""
    return 0.5 * (A + A.T)

def safe_solve(A, b):
    """
    Solve A x = b robustly.
    Falls back to pseudo-inverse if A is singular/ill-conditioned.
    """
    try:
        return np.linalg.solve(A, b)
    except np.linalg.LinAlgError:
        return np.linalg.pinv(A) @ b


# TRAIN-only R estimation
def estimate_R_from_train(df: pd.DataFrame) -> np.ndarray:
    """
    Estimate measurement noise covariance R using TRAIN only.
    Uses first-difference variance per hive as a proxy and aggregates via the median.
    """
    diags = []
    for _, g in df.groupby("tag_number"):
        z = g[OBS_COLS].to_numpy(float)
        if len(z) < 10:
            continue
        dz = np.diff(z, axis=0)
        d = 0.5 * np.nanvar(dz, axis=0)
        diags.append(d)

    if len(diags) == 0:
        raise ValueError("Could not estimate R from TRAIN (no usable hives).")

    R_diag = np.nanmedian(np.vstack(diags), axis=0)
    R_diag = np.where(np.isfinite(R_diag) & (R_diag > 1e-8), R_diag, 1e-3)
    R_diag = np.clip(R_diag, 1e-6, None)
    return np.diag(R_diag)


# dt / gap-aware scaling (linear in dt)

def dt_row_minutes(ts: np.ndarray) -> np.ndarray:
    """Compute per-row time delta in minutes (based on consecutive timestamps)."""
    ts = pd.to_datetime(ts, utc=True, errors="coerce").to_numpy()
    dt = np.full(len(ts), np.nan, float)
    if len(ts) >= 2:
        d = (ts[1:] - ts[:-1]) / np.timedelta64(1, "m")
        dt[1:] = d.astype(float)
    dt[0] = BASE_DT_MIN
    dt = np.where(np.isfinite(dt) & (dt > 0), dt, BASE_DT_MIN)
    return dt

def dt_since_last_observed_minutes(df_h: pd.DataFrame) -> np.ndarray:
    """
    Compute minutes since the most recent row that had ANY observed value.
    This makes dt grow over missing stretches, increasing process uncertainty.
    """
    has_any = df_h[OBS_COLS].notna().any(axis=1).to_numpy()
    ts = pd.to_datetime(df_h["published_at"], utc=True, errors="coerce").to_numpy()
    out = np.full(len(df_h), np.nan, float)
    last_t = None
    for i in range(len(df_h)):
        if last_t is None:
            out[i] = BASE_DT_MIN
        else:
            out[i] = (ts[i] - last_t) / np.timedelta64(1, "m")
        if has_any[i]:
            last_t = ts[i]
    out = np.where(np.isfinite(out) & (out > 0), out, BASE_DT_MIN)
    return out

def build_dt_scale(df_h: pd.DataFrame) -> np.ndarray | None:
    """
    Build a per-timestep scaling factor for Q based on time gaps.
    The scaling is linear in dt relative to BASE_DT_MIN and clipped to [DT_CLIP_MIN, DT_CLIP_MAX].
    """
    if not USE_DT_AWARE_Q:
        return None
    if DT_SCALE_MODE == "row_dt":
        dt = dt_row_minutes(df_h["published_at"].to_numpy())
    elif DT_SCALE_MODE == "since_observed":
        dt = dt_since_last_observed_minutes(df_h)
    else:
        raise ValueError(f"Unknown DT_SCALE_MODE: {DT_SCALE_MODE}")

    s = dt / BASE_DT_MIN
    s = np.where(np.isfinite(s) & (s > 0), s, 1.0)
    return np.clip(s, DT_CLIP_MIN, DT_CLIP_MAX)


# Chi-square quantiles for NIS calibration (raw NIS vs DOF)

def _chi2_ppf_wilson_hilferty(p: float, k: int) -> float:
    """Approximate chi-square inverse CDF using Wilson–Hilferty transform."""
    try:
        from scipy.stats import norm
        z = float(norm.ppf(p))
    except Exception:
        # Acklam inverse normal approximation
        def inv_norm_cdf(pp):
            a = [-3.969683028665376e+01, 2.209460984245205e+02,
                 -2.759285104469687e+02, 1.383577518672690e+02,
                 -3.066479806614716e+01, 2.506628277459239e+00]
            b = [-5.447609879822406e+01, 1.615858368580409e+02,
                 -1.556989798598866e+02, 6.680131188771972e+01,
                 -1.328068155288572e+01]
            c = [-7.784894002430293e-03, -3.223964580411365e-01,
                 -2.400758277161838e+00, -2.549732539343734e+00,
                 4.374664141464968e+00, 2.938163982698783e+00]
            d = [7.784695709041462e-03, 3.224671290700398e-01,
                 2.445134137142996e+00, 3.754408661907416e+00]
            plow = 0.02425
            phigh = 1 - plow
            if pp < plow:
                q = np.sqrt(-2 * np.log(pp))
                return (((((c[0] * q + c[1]) * q + c[2]) * q + c[3]) * q + c[4]) * q + c[5]) / \
                       ((((d[0] * q + d[1]) * q + d[2]) * q + d[3]) * q + 1)
            if pp > phigh:
                q = np.sqrt(-2 * np.log(1 - pp))
                return -(((((c[0] * q + c[1]) * q + c[2]) * q + c[3]) * q + c[4]) * q + c[5]) / \
                        ((((d[0] * q + d[1]) * q + d[2]) * q + d[3]) * q + 1)
            q = pp - 0.5
            r = q * q
            return (((((a[0] * r + a[1]) * r + a[2]) * r + a[3]) * r + a[4]) * r + a[5]) * q / \
                   (((((b[0] * r + b[1]) * r + b[2]) * r + b[3]) * r + b[4]) * r + 1)

        z = float(inv_norm_cdf(p))

    k = max(int(k), 1)
    return float(k * (1 - 2 / (9 * k) + z * np.sqrt(2 / (9 * k))) ** 3)

def chi2_ppf(p: float, k: int) -> float:
    """Prefer scipy.stats.chi2.ppf; otherwise use Wilson–Hilferty approximation."""
    try:
        from scipy.stats import chi2
        return float(chi2.ppf(p, df=int(k)))
    except Exception:
        return _chi2_ppf_wilson_hilferty(p, k)


# EKF model definition

def f_nonlinear(x: np.ndarray, alpha: float) -> np.ndarray:
    """Nonlinear process model: random-walk plus a small tanh nonlinearity."""
    return x + alpha * np.tanh(x)

def jacobian_F(x: np.ndarray, alpha: float) -> np.ndarray:
    """Jacobian of f_nonlinear with respect to x."""
    d = 1.0 + alpha * (1.0 - np.tanh(x) ** 2)
    return np.diag(d)

H_full = np.eye(D)

def ekf_run_fair(z: np.ndarray, Q_base: np.ndarray, R: np.ndarray, alpha: float, dt_scale=None):
    """
    Run an EKF with "fair" forecast evaluation:
      - x_pred[t] is recorded BEFORE assimilating z[t].
      - Update is performed only on observed dimensions at time t.
      - If dt_scale is provided, Q_base is scaled per time step (gap-aware optional).
    """
    T, d = z.shape
    assert d == D

    x = init_x0_from_first_valid(z)
    P = P0.copy()

    x_pred = np.zeros((T, d), float)
    x_filt = np.zeros((T, d), float)
    P_pred = np.zeros((T, d, d), float)

    nis = np.full((T,), np.nan, float)
    nis_norm = np.full((T,), np.nan, float)
    nis_dof = np.zeros((T,), int)

    I = np.eye(d)

    for t in range(T):
        scale = 1.0 if dt_scale is None else float(dt_scale[t])
        Q = Q_base * scale

        # Predict
        F = jacobian_F(x, alpha)
        xp = f_nonlinear(x, alpha)
        Pp = F @ P @ F.T + Q
        Pp = sym(Pp)

        x_pred[t] = xp
        P_pred[t] = Pp

        # Update (assimilate observed dimensions only)
        zt = z[t]
        m = np.isfinite(zt)
        m_dim = int(m.sum())
        nis_dof[t] = m_dim

        if m_dim > 0:
            Ht = H_full[m, :]
            Rt = R[np.ix_(m, m)]
            z_obs = zt[m]

            y = Ht @ xp
            r = z_obs - y

            S = Ht @ Pp @ Ht.T + Rt
            S = sym(S) + S_JITTER * np.eye(m_dim)

            # NIS (normalized innovation squared)
            Sinv_r = safe_solve(S, r)
            nis_t = float(r.T @ Sinv_r)
            nis[t] = nis_t
            nis_norm[t] = nis_t / max(m_dim, 1)

            # Kalman gain and posterior update
            K = (Pp @ Ht.T) @ safe_solve(S, np.eye(m_dim))

            x_new = xp + K @ r
            P_new = (I - K @ Ht) @ Pp @ (I - K @ Ht).T + K @ Rt @ K.T
            P_new = sym(P_new)

            x, P = x_new, P_new
        else:
            x, P = xp, Pp

        x_filt[t] = x

    return x_pred, x_filt, P_pred, nis, nis_norm, nis_dof


# Main pipeline

def main():
    print("Loading data splits...")
    train_df = load_and_clean(TRAIN_PATH)
    val_df = load_and_clean(VAL_PATH)
    test_df = load_and_clean(TEST_PATH)

    print("Train:", train_df.shape, "Val:", val_df.shape, "Test:", test_df.shape)
    print(
        "Hives (train/val/test):",
        train_df["tag_number"].nunique(),
        val_df["tag_number"].nunique(),
        test_df["tag_number"].nunique(),
    )

    # TRAIN-only normalization
    train_std = train_df[OBS_COLS].astype(float).std(skipna=True).to_numpy()
    train_std = np.where(np.isfinite(train_std) & (train_std > 1e-12), train_std, 1.0)

    def agg_zrmse(rmse_raw: np.ndarray) -> float:
        """Aggregate normalized RMSE across dimensions."""
        rmse_raw = np.asarray(rmse_raw, float)
        m = np.isfinite(rmse_raw) & np.isfinite(train_std) & (train_std > 1e-12)
        if m.sum() == 0:
            return np.nan
        return float(np.mean(rmse_raw[m] / train_std[m]))

    # TRAIN-only R
    R = estimate_R_from_train(train_df)
    R_diag = np.diag(R)
    print("\nEstimated R diag (TRAIN only):", R_diag)

   
    # Select hives from VAL only (no test-informed selection)
    
    usable_val_hives = sorted([
        int(h) for h in val_df["tag_number"].dropna().unique()
        if rows_with_any_obs(val_df[val_df["tag_number"].astype(int) == int(h)]) >= MIN_ROWS_VAL
    ])
    if len(usable_val_hives) == 0:
        raise ValueError("No usable VAL hives. Lower MIN_ROWS_VAL or verify your splits.")

    usable_test_hives = sorted([
        int(h) for h in test_df["tag_number"].dropna().unique()
        if rows_with_any_obs(test_df[test_df["tag_number"].astype(int) == int(h)]) >= MIN_ROWS_TEST
    ])

    print("\nUsable VAL hives (selection):", len(usable_val_hives))
    print("Usable TEST hives (reporting only):", len(usable_test_hives))

    pd.DataFrame({"hive_id": usable_val_hives}).to_csv(
        OUT_DIR / f"{MODEL_TAG}_eval_hives_val_only.csv", index=False
    )

    
    # Tune Q on VAL (VAL-only hive set)
    
    tuning_rows = []
    for s in Q_SCALES:
        Q_base = np.diag(np.clip(R_diag * float(s), 1e-8, None))
        agg_scores = []

        for hive_id in usable_val_hives:
            vh = val_df[val_df["tag_number"].astype(int) == hive_id].sort_values("published_at")
            z = vh[OBS_COLS].to_numpy(float)
            dt_scale = build_dt_scale(vh)

            xpred, *_ = ekf_run_fair(z, Q_base=Q_base, R=R, alpha=ALPHA, dt_scale=dt_scale)
            agg_scores.append(agg_zrmse(rmse_per_dim(z, xpred)))

        tuning_rows.append({
            "Q_scale": float(s),
            "median_agg_zrmse_val": float(np.nanmedian(np.array(agg_scores))),
            "mean_agg_zrmse_val": float(np.nanmean(np.array(agg_scores))),
            "num_hives_used": int(len(agg_scores)),
        })

    tuning_df = pd.DataFrame(tuning_rows).sort_values("median_agg_zrmse_val").reset_index(drop=True)
    best_scale = float(tuning_df.iloc[0]["Q_scale"])
    Q_best = np.diag(np.clip(R_diag * best_scale, 1e-8, None))

    print("\nQ tuning results (ranked, VAL-selected hives only):")
    print(tuning_df.to_string(index=False))
    print("\nBest Q_scale (VAL):", best_scale)

    tuning_df.to_csv(OUT_DIR / f"{MODEL_TAG}_q_tuning_results.csv", index=False)

   
    # Per-step exports for VAL + TEST
 
    def build_step_df(df_split: pd.DataFrame, split_name: str, hive_list) -> pd.DataFrame:
        """
        Run the EKF per hive and return a per-step dataframe for the given split.
        Only hives present in df_split are evaluated.
        """
        rows = []
        present = set(df_split["tag_number"].dropna().astype(int).unique())
        run_hives = [h for h in hive_list if h in present]

        for hive_id in run_hives:
            dh = df_split[df_split["tag_number"].astype(int) == int(hive_id)].sort_values("published_at")
            if dh.shape[0] == 0:
                continue

            t = pd.to_datetime(dh["published_at"].to_numpy(), utc=True)
            z = dh[OBS_COLS].to_numpy(float)

            dt_scale = build_dt_scale(dh)
            xpred, xfilt, Ppred, nis, nis_norm, nis_dof = ekf_run_fair(
                z, Q_base=Q_best, R=R, alpha=ALPHA, dt_scale=dt_scale
            )

            pred_var = np.maximum(np.diagonal(Ppred, axis1=1, axis2=2), 0.0)
            pred_std = np.sqrt(pred_var)

            rows.append(pd.DataFrame({
                "published_at": t,
                "hive_id": int(hive_id),
                "split": split_name,

                "y_true_temperature": z[:, 0],
                "y_true_humidity": z[:, 1],
                "y_true_audio_density": z[:, 2],

                "y_pred_temperature": xpred[:, 0],
                "y_pred_humidity": xpred[:, 1],
                "y_pred_audio_density": xpred[:, 2],

                "pred_std_temperature": pred_std[:, 0],
                "pred_std_humidity": pred_std[:, 1],
                "pred_std_audio_density": pred_std[:, 2],

                "ekf_filt_temperature": xfilt[:, 0],
                "ekf_filt_humidity": xfilt[:, 1],
                "ekf_filt_audio_density": xfilt[:, 2],

                "nis_raw": nis,
                "nis_norm": nis_norm,
                "nis_dof": nis_dof,

                "is_test_usable_hive": (int(hive_id) in set(usable_test_hives)),
            }))

        return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

    ekf_val_step = build_step_df(val_df, "val", usable_val_hives)
    ekf_test_step = build_step_df(test_df, "test", usable_val_hives)
    ekf_all = pd.concat([ekf_val_step, ekf_test_step], ignore_index=True)

    
    # Percentile-based anomaly thresholds + chi-square calibration thresholds
    
    # Percentile thresholds derived from VAL only (dof > 0)
    val_mask = (ekf_val_step["nis_dof"] > 0) & np.isfinite(ekf_val_step["nis_norm"].to_numpy(float))
    val_scores = ekf_val_step.loc[val_mask, "nis_norm"].to_numpy(float)

    thr95_pct = percentile_threshold(val_scores, 95.0, min_n=500)
    thr99_pct = percentile_threshold(val_scores, 99.0, min_n=500)

    print("\nVAL-only percentile thresholds (nis_norm):")
    print(" thr95_pct:", thr95_pct, " thr99_pct:", thr99_pct, " | samples:", int(np.isfinite(val_scores).sum()))

    def apply_thresholds(step_df: pd.DataFrame) -> pd.DataFrame:
        """
        Add anomaly flags (percentile-based) and consistency flags (chi-square-based)
        to a per-step dataframe.
        """
        step_df = step_df.copy()

        # Data-driven anomaly thresholds (applied to nis_norm)
        step_df["is_anomaly_norm_pct_p95"] = (
            (step_df["nis_dof"] > 0) &
            np.isfinite(step_df["nis_norm"]) &
            np.isfinite(thr95_pct) &
            (step_df["nis_norm"] > thr95_pct)
        ) if np.isfinite(thr95_pct) else np.zeros(len(step_df), dtype=bool)

        step_df["is_anomaly_norm_pct_p99"] = (
            (step_df["nis_dof"] > 0) &
            np.isfinite(step_df["nis_norm"]) &
            np.isfinite(thr99_pct) &
            (step_df["nis_norm"] > thr99_pct)
        ) if np.isfinite(thr99_pct) else np.zeros(len(step_df), dtype=bool)

        # Theoretical consistency thresholds (applied to nis_raw vs chi2(dof))
        step_df["chi2_thr_raw_p95"] = step_df["nis_dof"].apply(
            lambda k: chi2_ppf(0.95, int(k)) if int(k) > 0 else np.nan
        )
        step_df["chi2_thr_raw_p99"] = step_df["nis_dof"].apply(
            lambda k: chi2_ppf(0.99, int(k)) if int(k) > 0 else np.nan
        )

        step_df["is_inconsistent_chi2_p95"] = (
            (step_df["nis_dof"] > 0) &
            np.isfinite(step_df["nis_raw"]) &
            np.isfinite(step_df["chi2_thr_raw_p95"]) &
            (step_df["nis_raw"] > step_df["chi2_thr_raw_p95"])
        )

        step_df["is_inconsistent_chi2_p99"] = (
            (step_df["nis_dof"] > 0) &
            np.isfinite(step_df["nis_raw"]) &
            np.isfinite(step_df["chi2_thr_raw_p99"]) &
            (step_df["nis_raw"] > step_df["chi2_thr_raw_p99"])
        )

        return step_df

    ekf_val_step = apply_thresholds(ekf_val_step)
    ekf_test_step = apply_thresholds(ekf_test_step)
    ekf_all = apply_thresholds(ekf_all)

    # Save per-step files
    ekf_val_step.to_parquet(OUT_DIR / f"{MODEL_TAG}_forecasts_val.parquet", index=False)
    ekf_test_step.to_parquet(OUT_DIR / f"{MODEL_TAG}_forecasts_test.parquet", index=False)
    ekf_all.to_parquet(OUT_DIR / f"{MODEL_TAG}_val_test_forecast_nis.parquet", index=False)

    
    # Summary metrics per hive
   
    summary_rows = []
    calib_rows = []

    for split_name, df_step in [("val", ekf_val_step), ("test", ekf_test_step)]:
        for hive_id, g in df_step.groupby("hive_id"):
            z = g[["y_true_temperature", "y_true_humidity", "y_true_audio_density"]].to_numpy(float)
            pred = g[["y_pred_temperature", "y_pred_humidity", "y_pred_audio_density"]].to_numpy(float)
            base = persistence_missing_safe(z)

            rmse_base = rmse_per_dim(z, base)
            rmse_model = rmse_per_dim(z, pred)

            summary_rows.append({
                "hive_id": int(hive_id),
                "split": split_name,
                "baseline_agg_zrmse": agg_zrmse(rmse_base),
                "ekf_agg_zrmse": agg_zrmse(rmse_model),
                "baseline_rmse_temp": rmse_base[0],
                "baseline_rmse_humid": rmse_base[1],
                "baseline_rmse_audio": rmse_base[2],
                "ekf_rmse_temp": rmse_model[0],
                "ekf_rmse_humid": rmse_model[1],
                "ekf_rmse_audio": rmse_model[2],
                "n_rows": int(len(g)),
                "is_test_usable_hive": bool(g["is_test_usable_hive"].iloc[0]),
            })

            # Calibration exceedance rates (chi-square based)
            m95 = (g["nis_dof"] > 0) & np.isfinite(g["nis_raw"]) & np.isfinite(g["chi2_thr_raw_p95"])
            m99 = (g["nis_dof"] > 0) & np.isfinite(g["nis_raw"]) & np.isfinite(g["chi2_thr_raw_p99"])
            calib_rows.append({
                "hive_id": int(hive_id),
                "split": split_name,
                "chi2_exceed_rate_p95": float(np.mean(g.loc[m95, "nis_raw"] > g.loc[m95, "chi2_thr_raw_p95"])) if m95.any() else np.nan,
                "chi2_exceed_rate_p99": float(np.mean(g.loc[m99, "nis_raw"] > g.loc[m99, "chi2_thr_raw_p99"])) if m99.any() else np.nan,
                "n_dof_pos": int((g["nis_dof"] > 0).sum()),
                "is_test_usable_hive": bool(g["is_test_usable_hive"].iloc[0]),
            })

    summary_metrics = pd.DataFrame(summary_rows)
    summary_metrics.to_csv(OUT_DIR / f"{MODEL_TAG}_summary_metrics_per_hive.csv", index=False)

    calib_df = pd.DataFrame(calib_rows)
    calib_df.to_csv(OUT_DIR / f"{MODEL_TAG}_nis_chi2_calibration_per_hive.csv", index=False)

    # Overall calibration summary per split
    overall_calib = []
    for split_name, df_step in [("val", ekf_val_step), ("test", ekf_test_step)]:
        g = df_step
        m95 = (g["nis_dof"] > 0) & np.isfinite(g["nis_raw"]) & np.isfinite(g["chi2_thr_raw_p95"])
        m99 = (g["nis_dof"] > 0) & np.isfinite(g["nis_raw"]) & np.isfinite(g["chi2_thr_raw_p99"])
        overall_calib.append({
            "split": split_name,
            "chi2_exceed_rate_p95_overall": float(np.mean(g.loc[m95, "nis_raw"] > g.loc[m95, "chi2_thr_raw_p95"])) if m95.any() else np.nan,
            "chi2_exceed_rate_p99_overall": float(np.mean(g.loc[m99, "nis_raw"] > g.loc[m99, "chi2_thr_raw_p99"])) if m99.any() else np.nan,
            "n_samples_dof_pos": int((g["nis_dof"] > 0).sum()),
        })
    pd.DataFrame(overall_calib).to_csv(OUT_DIR / f"{MODEL_TAG}_nis_chi2_calibration_overall.csv", index=False)

    # Run configuration for reproducibility
    pd.DataFrame([{
        "MODEL_TAG": MODEL_TAG,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "OBS_COLS": ",".join(OBS_COLS),
        "D": D,
        "alpha": ALPHA,
        "P0_diag": ",".join([f"{x:.6g}" for x in np.diag(P0)]),
        "best_Q_scale": best_scale,
        "S_JITTER": S_JITTER,

        "USE_DT_AWARE_Q": USE_DT_AWARE_Q,
        "DT_SCALE_MODE": DT_SCALE_MODE,
        "DT_CLIP_MIN": DT_CLIP_MIN,
        "DT_CLIP_MAX": DT_CLIP_MAX,

        # Percentile anomaly thresholds (VAL-derived)
        "nis_norm_pct_p95_val": thr95_pct,
        "nis_norm_pct_p99_val": thr99_pct,
        "val_nis_samples_used": int(np.isfinite(val_scores).sum()),

        "MIN_ROWS_VAL": MIN_ROWS_VAL,
        "MIN_ROWS_TEST": MIN_ROWS_TEST,
        "val_selected_hives": len(usable_val_hives),
        "test_usable_hives_reporting_only": len(usable_test_hives),
        "seed": 42,
    }]).to_csv(OUT_DIR / f"{MODEL_TAG}_run_config.csv", index=False)

    print("\nSaved outputs to:", OUT_DIR)
    print(" -", OUT_DIR / f"{MODEL_TAG}_eval_hives_val_only.csv")
    print(" -", OUT_DIR / f"{MODEL_TAG}_q_tuning_results.csv")
    print(" -", OUT_DIR / f"{MODEL_TAG}_forecasts_val.parquet")
    print(" -", OUT_DIR / f"{MODEL_TAG}_forecasts_test.parquet")
    print(" -", OUT_DIR / f"{MODEL_TAG}_val_test_forecast_nis.parquet")
    print(" -", OUT_DIR / f"{MODEL_TAG}_summary_metrics_per_hive.csv")
    print(" -", OUT_DIR / f"{MODEL_TAG}_nis_chi2_calibration_per_hive.csv")
    print(" -", OUT_DIR / f"{MODEL_TAG}_nis_chi2_calibration_overall.csv")
    print(" -", OUT_DIR / f"{MODEL_TAG}_run_config.csv")
    print("\nCompleted EKF pipeline: VAL-only hive selection, percentile anomaly flags, chi-square calibration, and dt-aware Q scaling.")

if __name__ == "__main__":
    main()

Loading data splits...
Train: (560733, 31) Val: (120160, 31) Test: (120187, 31)
Hives (train/val/test): 52 52 52

Estimated R diag (TRAIN only): [0.0124549  0.29751825 2.82167299]

Usable VAL hives (selection): 33
Usable TEST hives (reporting only): 27

Q tuning results (ranked, VAL-selected hives only):
 Q_scale  median_agg_zrmse_val  mean_agg_zrmse_val  num_hives_used
    3.00              0.169828            0.185581              33
    1.00              0.172389            0.186541              33
    0.30              0.187355            0.197785              33
    0.10              0.219669            0.223076              33
    0.03              0.278039            0.272877              33
    0.01              0.344936            0.339546              33

Best Q_scale (VAL): 3.0

VAL-only percentile thresholds (nis_norm):
 thr95_pct: 2.6507522801453245  thr99_pct: 7.73513773049664  | samples: 57425

Saved outputs to: ..\backend\data\EKF_Outputs
 - ..\backend\data\EKF_Outputs\